<a href="https://colab.research.google.com/github/yzhang-data/Fraud_Detection/blob/main/Fraud_Detection_02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
To pre-process data to ensure it is model-ready. This involves data cleaning, feature engineering, and splitting the data into training and testing sets.

## 1. Load Data
Based on your previous EDA, we'll now focus on cleaning the data and creating new variables for model training.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# define file path
file_path='/content/drive/My Drive/Fraud/transactions.txt'

In [ ]:
#load package
import pandas as pd
import seaborn as sns
import numpy as np
import warnings
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None) # pandas setting

In [ ]:
drop_cols = [
    "posOnPremises",
    "recurringAuthInd",
    "merchantState",
    "merchantZip",
    "echoBuffer",
    "merchantCity"
]

date_cols = [
    "transactionDateTime",
    "currentExpDate",
    "accountOpenDate",
    "dateOfLastAddressChange"
]

num_cols = [
    "creditLimit",
    "availableMoney",
    "transactionAmount",
    "currentBalance"
]

cat_cols = [
    "merchantCategoryCode",
    "transactionType",
    "acqCountry",
    "merchantCountryCode",
    "posEntryMode",
    "posConditionCode"
]

log_cols = [
    "transactionAmount",
    "availableMoney",
    "currentBalance"
]

high_card_cols = [
    "merchantName",
    "customerId"]

In [ ]:
# read data
try:
    df = (pd.read_json(file_path, lines=True)
      .drop(columns=drop_cols))
    df = df.replace(r"^\s*$", pd.NA, regex=True)
    print(f"Dataset loaded successfully from google drive")
except FileNotFoundError:
    print(f"Error: The file was not found. Please verify and retry.")
    df = None
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")
    df = None
df.head(3)

Dataset loaded successfully from google drive


,accountNumber,customerId,creditLimit,availableMoney,transactionDateTime,transactionAmount,merchantName,acqCountry,merchantCountryCode,posEntryMode,posConditionCode,merchantCategoryCode,currentExpDate,accountOpenDate,dateOfLastAddressChange,cardCVV,enteredCVV,cardLast4Digits,transactionType,currentBalance,cardPresent,expirationDateKeyInMatch,isFraud
0,737265056,737265056,5000,5000.0,2016-08-13T14:27:32,98.55,Uber,US,US,02,01,rideshare,06/2023,2015-03-14,2015-03-14,414,414,1803,PURCHASE,0.0,False,False,False
1,737265056,737265056,5000,5000.0,2016-10-11T05:05:54,74.51,AMC #191138,US,US,09,01,entertainment,02/2024,2015-03-14,2015-03-14,486,486,767,PURCHASE,0.0,True,False,False
2,737265056,737265056,5000,5000.0,2016-11-08T09:18:39,7.47,Play Store,US,US,09,01,mobileapps,08/2025,2015-03-14,2015-03-14,486,486,767,PURCHASE,0.0,False,False,False


## Missing

In [ ]:
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

## Create new variables

In [ ]:
def create_new_var(df,date_cols,log_cols):

  # Convert dates
  for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

  # Transaction time features
  df["transaction_year"] = df["transactionDateTime"].dt.year
  df["transaction_month"] = df["transactionDateTime"].dt.month
  df["transaction_day"] = df["transactionDateTime"].dt.day
  df["transaction_hour"] = df["transactionDateTime"].dt.hour
  df["transaction_weekday"] = df["transactionDateTime"].dt.weekday


  # Weekend indicator
  df["is_weekend"] = (
      df["transaction_weekday"].isin([5, 6])
      ).astype(int)

  # Account age
  df["account_age_days"] = (
      df["transactionDateTime"] -
      df["accountOpenDate"]
      ).dt.days

  # Days since address change
  df["days_since_address_change"] = (
      df["transactionDateTime"] -
      df["dateOfLastAddressChange"]
      ).dt.days

  # Recent address change indicator
  df["recent_address_change_30d"] = (
      df["days_since_address_change"] <= 30
      ).astype(int)

  # Days until card expiration
  df["days_to_expiry"] = (
      df["currentExpDate"] -
      df["transactionDateTime"]
      ).dt.days

  df["amount_to_available_ratio"] = (
      df["transactionAmount"] /
      df["availableMoney"].replace(0, np.nan))

  df["amount_to_credit_ratio"] = (
      df["transactionAmount"] /
      df["creditLimit"])

  df["credit_utilization"] = (
      df["currentBalance"] /
      df["creditLimit"]
      )

  df["insufficient_available_money"] = (
      df["transactionAmount"] > df["availableMoney"]
      ).astype(int)

  # Handle infinite values
  ratio_cols = [
      "amount_to_credit_ratio",
      "amount_to_available_ratio",
      "credit_utilization"
      ]

  df[ratio_cols] = (
        df[ratio_cols]
        .replace([np.inf, -np.inf], np.nan))

# =====================================================
# Verification Features
# =====================================================

  df["cvv_match"] = (
      df["cardCVV"] ==
      df["enteredCVV"]).astype(int)

  # Binary existing features
  df["cardPresent"] = df["cardPresent"].astype(int)
  df["expirationDateKeyInMatch"] = (
      df["expirationDateKeyInMatch"].astype(int)
      )

  for col in log_cols:
      df[f"log_{col}"] = np.log1p(
      df[col].clip(lower=0))

  new_features = [
      "transaction_year",
      "transaction_month",
      "transaction_day",
      "transaction_hour",
      "transaction_weekday",
      "is_weekend",
      "account_age_days",
      "days_since_address_change",
      "recent_address_change_30d",
      "days_to_expiry",
      "amount_to_available_ratio",
      "amount_to_credit_ratio",
      "credit_utilization",
      "insufficient_available_money",
      "cvv_match",
      "cardPresent",
      "expirationDateKeyInMatch",
      "log_transactionAmount",
      "log_availableMoney",
      "log_currentBalance"]

  return df, new_features

In [ ]:
df, new_features=create_new_var(df,date_cols,log_cols)

/tmp/ipykernel_563/2352044231.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


Fraud Detection Practice
*   Customer History Features

*   Historical behavioral features were computed using only transactions that occurred before the current transaction by sorting transactions chronologically within each customer. This design avoids look-ahead bias and better reflects a real-world fraud detection scenario.



In [ ]:
def create_customer_history_features(df):
  """ Create customer history features. """

  # Sort transactions chronologically within each customer
  df = (
      df.sort_values(
          ["customerId", "transactionDateTime"]
          ).reset_index(drop=True)
          )

  # -----------------------------------------------------
  # Number of previous transactions
  # -----------------------------------------------------

  df["customer_txn_count"] = (
      df.groupby("customerId")
        .cumcount())

  # First transaction indicator
  df["first_transaction"] = (
      df["customer_txn_count"] == 0).astype(int)

  # -----------------------------------------------------
  # Time since previous transaction
  # -----------------------------------------------------

  df["hours_since_last_txn"] = (
      df.groupby("customerId")["transactionDateTime"]
        .diff()
        .dt.total_seconds() / 3600)

  # -----------------------------------------------------
  # Previous transaction amount
  # -----------------------------------------------------

  df["prev_transaction_amount"] = (
      df.groupby("customerId")["transactionAmount"]
        .shift(1))

  # -----------------------------------------------------
  # Historical average transaction amount
  # (excluding current transaction)
  # -----------------------------------------------------

  hist_sum = (
      df.groupby("customerId")["transactionAmount"]
        .cumsum()
        - df["transactionAmount"])

  hist_count = (
      df.groupby("customerId")
        .cumcount())

  df["customer_avg_amount"] = (
      hist_sum /
      hist_count.replace(0, np.nan))


  # -----------------------------------------------------
  # Current amount relative to historical average
  # -----------------------------------------------------

  df["amount_vs_history"] = (
      df["transactionAmount"] /
      df["customer_avg_amount"])

  df["amount_vs_history"] = (
      df["amount_vs_history"]
      .replace([np.inf, -np.inf], np.nan))

  # =====================================================
  # Customer-Merchant History
  # =====================================================

  # Previous transactions with the same merchant
  df["customer_merchant_txn_count"] = (
      df.groupby(["customerId", "merchantName"]
                 ).cumcount())

  # Time since previous transaction at same merchant
  df["seconds_since_customer_merchant_txn"] = (
      df.groupby(
          ["customerId", "merchantName"]
          )["transactionDateTime"]
      .diff()
      .dt.total_seconds())

  # Previous amount at same merchant
  df["previous_customer_merchant_amount"] = (
      df.groupby(
          ["customerId", "merchantName"]
          )["transactionAmount"]
      .shift(1)
      )

  # Same amount as previous transaction at same merchant
  df["same_amount_as_previous_merchant_txn"] = (
      df["transactionAmount"]
      ==
      df["previous_customer_merchant_amount"]).astype(int)

  # Possible multi-swipe behavior
  df["rapid_repeat_transaction"] = (
      df["seconds_since_customer_merchant_txn"]
      .le(180)).astype(int)

  # Combined signal
  df["possible_multi_swipe"] = (
      (df["rapid_repeat_transaction"] == 1) &
       (df["same_amount_as_previous_merchant_txn"] == 1)
       ).astype(int)

  return df

In [ ]:
df = create_customer_history_features(df)

In [ ]:
# verify the data
df[[
"customer_txn_count",
"hours_since_last_txn",
"customer_avg_amount",
"amount_vs_history",
"customer_merchant_txn_count",
     "previous_customer_merchant_amount"]].head()

,customer_txn_count,hours_since_last_txn,customer_avg_amount,amount_vs_history,customer_merchant_txn_count,previous_customer_merchant_amount
0,0,NaN,NaN,NaN,0,NaN
1,1,18.830278,205.130000,0.226344,0,NaN
2,2,115.868056,125.780000,3.010574,0,NaN
3,3,388.258889,210.076667,0.314504,0,NaN
4,4,8.860000,174.075000,0.813442,0,NaN


In [ ]:
# quick check for new potential variable
df.groupby(
    "possible_multi_swipe"
)["isFraud"].agg(
    Count="count",
    Fraud_Rate="mean"
)

,Count,Fraud_Rate
possible_multi_swipe,,
0,772978,0.015762
1,13385,0.017408


Transactions identified as possible multi-swipe events (same customer, same merchant, same amount within 3 minutes) exhibit a slightly higher fraud rate than other transactions. Although the difference is modest, this feature may capture suspicious repeated payment behavior and is retained for modeling.

In [ ]:
pd.crosstab(
    df["possible_multi_swipe"],
    df["transactionType"],
    normalize="index"
) * 100

transactionType,ADDRESS_VERIFICATION,PURCHASE,REVERSAL,Unknown
possible_multi_swipe,,,,
0,2.580539,95.443725,1.886470,0.089265
1,1.658573,55.539783,42.741875,0.059768


Possible multi-swipe transactions are strongly associated with reversal transactions. Nearly 43% of transactions flagged as possible multi-swipes are reversals, compared with less than 2% among other transactions. However, the fraud rate increases only slightly, suggesting that while repeated transactions are common in reversal events, most reversals are legitimate rather than fraudulent. Therefore, this feature is retained as a behavioral indicator for modeling.

In [ ]:
pd.crosstab(
    df["possible_multi_swipe"],
    df["transactionType"],
    normalize="columns"
)

transactionType,ADDRESS_VERIFICATION,PURCHASE,REVERSAL,Unknown
possible_multi_swipe,,,,
0,0.988993,0.990024,0.718219,0.988539
1,0.011007,0.009976,0.281781,0.011461


Possible multi-swipe transactions are much more common among reversal transactions (28%) than purchase transactions (1%), indicating that the feature effectively captures repeated payment behavior. Although its direct association with fraud is relatively weak, it provides useful behavioral information and is retained for modeling.

In [ ]:
df.groupby(
    "rapid_repeat_transaction"
)["isFraud"].agg(
    Count="count",
    Fraud_Rate="mean"
)

,Count,Fraud_Rate
rapid_repeat_transaction,,
0,771375,0.015741
1,14988,0.018348


## Train, Test Split


*   Option 1: time split
    *   Realistic
    *   Lower risk of data leakage

*   Option 2: random split
    *   Fraud rate might be more stable
    *   Simple
    *   Higher risk of data leakage

### Option 1: time split
cutoff_date = df["transactionDateTime"].quantile(0.8)

train = df[
    df["transactionDateTime"] <= cutoff_date
]

test = df[
    df["transactionDateTime"] > cutoff_date
]

### Option 2: random split
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["isFraud"],
    random_state=42)

In [ ]:
# =====================================================
# Train/Test Split (Time-based)
# =====================================================

# Make sure data is sorted by transaction time
df = (
    df.sort_values("transactionDateTime")
      .reset_index(drop=True)
)


# Use 80% earliest transactions as train
split_date = df["transactionDateTime"].quantile(0.8)


train = df[
    df["transactionDateTime"] <= split_date
].copy()


test = df[
    df["transactionDateTime"] > split_date
].copy()


print("Train shape:", train.shape)
print("Test shape:", test.shape)


print("\nTrain fraud rate:")
print(train["isFraud"].mean())


print("\nTest fraud rate:")
print(test["isFraud"].mean())


Train shape: (629090, 53)
Test shape: (157273, 53)

Train fraud rate:
0.016061294886264288

Test fraud rate:
0.01470691091287125


In [ ]:
# =====================================================
# Separate X and y
# =====================================================

target = "isFraud"


X_train = train.drop(columns=[target])
y_train = train[target]


X_test = test.drop(columns=[target])
y_test = test[target]


print(X_train.shape)
print(X_test.shape)

(629090, 52)
(157273, 52)


In [ ]:
# =====================================================
# One-hot encoding
# =====================================================

X_train = pd.get_dummies(
    X_train,
    columns=cat_cols,
    drop_first=True
)


X_test = pd.get_dummies(
    X_test,
    columns=cat_cols,
    drop_first=True
)


# Align columns
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

In [ ]:
high_card_cols = [
    "merchantName",
    "customerId"]

In [ ]:
# =====================================================
# Frequency Encoding
# =====================================================

for col in high_card_cols:

    freq_map = (
        X_train[col]
        .value_counts()
        .to_dict()
    )

    X_train[f"{col}_freq"] = (
        X_train[col]
        .map(freq_map)
    )

    X_test[f"{col}_freq"] = (
        X_test[col]
        .map(freq_map)
        .fillna(0)
    )

## Drop Raw Columns

In [ ]:
raw_cols = [
    "transactionDateTime",
    "currentExpDate",
    "accountOpenDate",
    "dateOfLastAddressChange",
    "cardCVV",
    "enteredCVV",
    "cardLast4Digits",
    "accountNumber"] + cat_cols + high_card_cols

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(
    X_train.isna().sum().sort_values(ascending=False).head(10))


(629090, 77)
(157273, 77)
prev_transaction_amount     4972
availableMoney                 0
transactionAmount              0
cardLast4Digits                0
creditLimit                    0
cardPresent                    0
expirationDateKeyInMatch       0
transaction_year               0
transaction_month              0
transaction_day                0
dtype: int64


In [ ]:
X_train["has_customer_merchant_history"] = (
    X_train["previous_customer_merchant_amount"].notna()
).astype(int)

X_test["has_customer_merchant_history"] = (
    X_test["previous_customer_merchant_amount"].notna()
).astype(int)


In [ ]:
# NaN from history features
history_cols = [
    "previous_customer_merchant_amount",
    "seconds_since_customer_merchant_txn",
    "amount_vs_history",
    "hours_since_last_txn",
    "customer_avg_amount",
    "prev_transaction_amount"
]

X_train[history_cols] = (
    X_train[history_cols]
    .fillna(-1)
)

X_test[history_cols] = (
    X_test[history_cols]
    .fillna(-1)
)

In [ ]:
bool_cols = X_train.select_dtypes(include="bool").columns

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [ ]:
# final check
X_train.isna().sum().sum()

np.int64(0)

In [ ]:
# final check
X_train.select_dtypes(include="object").columns.tolist()

[]

In [ ]:
# final check
print((X_train.columns == X_test.columns).all())

True


*   Expectation:

    *   Index([], dtype='object')

    *   np.int64(0)




*   customerId_freq might not have any additional information

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

isFraud
False    0.983939
True     0.016061
Name: proportion, dtype: float64
isFraud
False    0.985293
True     0.014707
Name: proportion, dtype: float64


## Save Data

In [ ]:
import os

os.getcwd()

'/content'

In [ ]:
save_path = "/content/drive/MyDrive/Fraud/"

In [ ]:
X_train.to_csv(
    save_path + "X_train_processed.csv",
    index=False
)

X_test.to_csv(
    save_path + "X_test_processed.csv",
    index=False
)

y_train.to_csv(
    save_path + "y_train.csv",
    index=False
)

y_test.to_csv(
    save_path + "y_test.csv",
    index=False
)

In [ ]:
import json
import os

save_path = "/content/drive/My Drive/Fraud/processed_data/"

# Create the directory if it does not exist
os.makedirs(save_path, exist_ok=True)

with open(
    save_path + "model_columns.json",
    "w"
) as f:
    json.dump(
        X_train.columns.tolist(),
        f
    )

In [ ]:
feature_info = pd.DataFrame({
    "feature": X_train.columns,
    "dtype": X_train.dtypes.values
})

feature_info.to_csv(
    save_path + "feature_info.csv",
    index=False
)